In [ ]:
import os
import subprocess

SRC_LANG = 'hsb'
TRG_LANG = 'de'
BUCC_SETS = ['train', 'test']
MODEL_NAME = 'glot500'

DATA_DIR = '../data'

# NOTE: change 'results_full_MODEL' folder name or delete it before executing this file again to ensure a fresh folder before execution
RESULTS_DIR = f'../all_executions/re-computation_folder/results_full_{MODEL_NAME}'
EMBEDDINGS_DIR = os.path.join(RESULTS_DIR, 'new_embeddings', 'doc', 'bucc2017', f'{SRC_LANG}-{TRG_LANG}')
DICTIONARIES_DIR = os.path.join(RESULTS_DIR, 'dictionaries')
MINING_DIR = os.path.join(RESULTS_DIR, 'mining', 'bucc2017', f'{SRC_LANG}-{TRG_LANG}')

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(DICTIONARIES_DIR, exist_ok=True)
os.makedirs(MINING_DIR, exist_ok=True)

print("-" * 50)

for data_split in BUCC_SETS:
    for lang in [SRC_LANG, TRG_LANG]:
        input_file = f"{DATA_DIR}/bucc_style_data/{SRC_LANG}-{TRG_LANG}/{SRC_LANG}-{TRG_LANG}.{data_split}.{lang}"
        output_file = f"{EMBEDDINGS_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.{lang}.vec"
        
        command = f"python ../code/contextual_sentence_embeddings.py --input_file {input_file} --output_file {output_file} -m {MODEL_NAME}"
        
        subprocess.run(command, shell=True, check=True)

print("-" * 50)

In [ ]:
import os
import subprocess

for data_split in BUCC_SETS:
    source_vec_file = f"{EMBEDDINGS_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.{SRC_LANG}.vec"
    target_vec_file = f"{EMBEDDINGS_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.{TRG_LANG}.vec"
    
    output_sim_file = f"{DICTIONARIES_DIR}/DOC.{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim"
    
    command = f"python ../code/scripts/bilingual_nearest_neighbor.py --source_embeddings {source_vec_file} --target_embeddings {target_vec_file} --output {output_sim_file} --knn 10 -m csls --cslsknn 20"

    subprocess.run(command, shell=True, check=True)

In [ ]:
import os
import subprocess

for data_split in BUCC_SETS:
    sim_file = f"{DICTIONARIES_DIR}/DOC.{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim"
    
    pred_file = f"{MINING_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim.pred"
    gold_file = f"{DATA_DIR}/bucc_style_data/{SRC_LANG}-{TRG_LANG}/{SRC_LANG}-{TRG_LANG}.{data_split}.gold"
    res_file = f"{MINING_DIR}/{MODEL_NAME}.{SRC_LANG}-{TRG_LANG}.{data_split}.sim.pred.res"

    filter_command = f"python ../code/scripts/filter.py --input {sim_file} --output {pred_file} -m static -th -1.0"
    subprocess.run(filter_command, shell=True, check=True)
    
    eval_command = f"python ../code/scripts/bucc_f-score.py -p {pred_file} -g {gold_file} > {res_file}"
    subprocess.run(eval_command, shell=True, check=True)

    with open(res_file, 'r') as f:
        print(f.read())